# 2026년 3월 일별 주요지분 현황 DB 생성

**데이터 출처**: [KRX 상장법인 지분정보센터](https://fair.krx.co.kr/p/Fids0103/)  
**수집 기간**: 2026년 3월 2일(월) ~ 3월 13일(금)  
**다운로드 파라미터**: `strtdd=YYYY-MM-DD`  
**비고**: 주말·공휴일은 데이터 없음 (컬럼명만 포함)


In [ ]:
# 최초 1회: 필요 라이브러리 설치
# !pip install requests pandas openpyxl

In [ ]:
import requests
import pandas as pd
import io
import time
import re
from datetime import date, timedelta

print(f"pandas  : {pd.__version__}")
print(f"requests: {requests.__version__}")

## 1. 설정 및 날짜 범위

In [ ]:
BASE_URL = "https://fair.krx.co.kr"
PAGE_URL = f"{BASE_URL}/p/Fids0103/"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8",
    "Referer": PAGE_URL,
}

# 조회 기간
START = date(2026, 3, 2)
END   = date(2026, 3, 13)

def get_weekdays(start: date, end: date) -> list:
    """시작~종료 사이 평일(월~금) 목록"""
    days, cur = [], start
    while cur <= end:
        if cur.weekday() < 5:
            days.append(cur)
        cur += timedelta(days=1)
    return days

target_days = get_weekdays(START, END)
DOW = ['월','화','수','목','금','토','일']
print("조회 대상 날짜:")
for d in target_days:
    print(f"  {d.strftime('%Y-%m-%d')} ({DOW[d.weekday()]})")

## 2. 세션 초기화 및 다운로드 엔드포인트 자동 탐색

In [ ]:
session = requests.Session()
session.headers.update(HEADERS)

# ── 메인 페이지 로드: 세션 쿠키 획득 ──────────────────────────────
print("메인 페이지 로딩 중...")
resp = session.get(PAGE_URL, timeout=30)
print(f"HTTP 상태: {resp.status_code}  |  인코딩: {resp.encoding}")
html = resp.text

In [ ]:
# ── JS에서 다운로드 엔드포인트 탐색 ──────────────────────────────
# 'strtdd' 파라미터를 포함하는 URL 패턴 검색

found_endpoints = set()

# 1) 인라인 HTML에서 탐색
hits = re.findall(
    r'["\']([^"\']*/[^"\']*)["\'\s]',
    html
)
for h in hits:
    if any(kw in h.lower() for kw in ['excel','down','export','strtdd','fids']):
        found_endpoints.add(h.strip())

# 2) 링크된 JS 파일에서 탐색
js_urls = re.findall(r'src=["\']([^"\']*.js[^"\']*)["\'\s>]', html)
for js_url in js_urls:
    if not js_url.startswith('http'):
        js_url = BASE_URL + ('/' if not js_url.startswith('/') else '') + js_url
    try:
        jr = session.get(js_url, timeout=10)
        if jr.status_code == 200:
            for h in re.findall(r'["\']([^"\']*/[^"\']*)["\'\s,;]', jr.text):
                if any(kw in h.lower() for kw in ['excel','down','export','strtdd','fids']):
                    found_endpoints.add(h.strip())
    except Exception:
        pass

print("발견한 관련 엔드포인트:")
for ep in sorted(found_endpoints):
    print(" ", ep)

## 3. 날짜별 Excel 다운로드

**파라미터**: `strtdd=YYYY-MM-DD` (브라우저 Network 탭에서 확인한 실제 파라미터명)

아래 셀에서 `DOWNLOAD_URL`을 위의 탐색 결과나 브라우저 DevTools에서 확인한 실제 URL로 교체하세요.

In [ ]:
# ──────────────────────────────────────────────────────────────────
# ★ 브라우저 Network 탭에서 확인한 실제 다운로드 URL로 교체하세요 ★
#
# 확인 방법:
#   1) 브라우저에서 https://fair.krx.co.kr/p/Fids0103/ 접속
#   2) F12 → Network 탭 → XHR/Fetch 필터
#   3) 날짜 선택 후 [Excel 다운로드] 클릭
#   4) Network 탭에서 요청 URL 전체 복사
# ──────────────────────────────────────────────────────────────────

# 후보 URL 목록 (탐색 결과 또는 직접 확인한 URL)
CANDIDATE_URLS = [
    f"{BASE_URL}/p/Fids0103/excel",
    f"{BASE_URL}/p/Fids0103/excelDown",
    f"{BASE_URL}/p/Fids0103/download",
    f"{BASE_URL}/comm/fileDn/Fids0103/excelDown.cmd",
    f"{BASE_URL}/comm/bfebpbas/CFIDS0103.cmd",
    # 탐색으로 발견된 URL을 아래에 추가
    # *found_endpoints  ← 위 셀 실행 후 여기에 붙여넣기
]

# 실제로 동작하는 다운로드 URL 자동 검증
TEST_DATE = "2026-03-03"  # 테스트용 날짜 (화요일)

DOWNLOAD_URL = None
DATE_PARAM   = "strtdd"   # 브라우저에서 확인된 파라미터명

print("다운로드 URL 후보 검증 중...")
for url in CANDIDATE_URLS:
    try:
        r = session.get(url, params={DATE_PARAM: TEST_DATE}, timeout=15)
        ct = r.headers.get('Content-Type', '')
        print(f"  {url}")
        print(f"    → 상태: {r.status_code}  Content-Type: {ct[:60]}")
        if r.status_code == 200 and any(
            k in ct for k in ['excel','spreadsheet','octet','download']
        ):
            DOWNLOAD_URL = url
            print(f"    ✓ Excel 응답 확인! 이 URL 사용")
            break
        # POST 시도
        rp = session.post(url, data={DATE_PARAM: TEST_DATE}, timeout=15)
        ct2 = rp.headers.get('Content-Type', '')
        if rp.status_code == 200 and any(
            k in ct2 for k in ['excel','spreadsheet','octet','download']
        ):
            DOWNLOAD_URL = url
            DATE_PARAM   = DATE_PARAM  # POST 방식
            print(f"    ✓ POST로 Excel 응답 확인! 이 URL 사용")
            break
    except Exception as e:
        print(f"  {url} → 오류: {e}")

if DOWNLOAD_URL:
    print(f"\n최종 다운로드 URL: {DOWNLOAD_URL}")
else:
    print("\n⚠ 자동 탐색 실패. 아래 셀에서 DOWNLOAD_URL을 수동으로 지정하세요.")

In [ ]:
# ── 수동 지정 (자동 탐색 실패 시 여기서 직접 입력) ───────────────
# DOWNLOAD_URL = "https://fair.krx.co.kr/???"  # 브라우저 Network 탭 확인 후 입력
# DATE_PARAM   = "strtdd"
# ─────────────────────────────────────────────────────────────────

print(f"사용할 URL   : {DOWNLOAD_URL}")
print(f"날짜 파라미터: {DATE_PARAM}")

## 4. 전체 날짜 데이터 수집

In [ ]:
def fetch_daily_excel(target_date: date, url: str, param: str) -> pd.DataFrame:
    """
    하루치 Excel 다운로드 → DataFrame 반환.
    데이터 없는 날(주말·공휴일)은 빈 DataFrame 반환.
    """
    date_str = target_date.strftime("%Y-%m-%d")  # strtdd 형식
    params   = {param: date_str}

    # GET 시도
    r = session.get(url, params=params, timeout=60)
    if r.status_code != 200:
        # POST 시도
        r = session.post(url, data=params, timeout=60)
    r.raise_for_status()

    ct = r.headers.get('Content-Type', '')
    if not any(k in ct for k in ['excel','spreadsheet','octet']):
        # JSON 혹은 HTML 응답이면 빈 DataFrame
        return pd.DataFrame()

    try:
        df = pd.read_excel(io.BytesIO(r.content), dtype=str)
        # 실질 데이터 없는 파일(컬럼만): 행이 0개
        return df
    except Exception:
        return pd.DataFrame()


all_frames = []
DOW = ['월','화','수','목','금','토','일']

for d in target_days:
    label = f"{d.strftime('%Y-%m-%d')}({DOW[d.weekday()]})"
    print(f"[{label}] 수집 중...", end=" ")
    try:
        df = fetch_daily_excel(d, DOWNLOAD_URL, DATE_PARAM)
        if df.empty:
            print("데이터 없음 (주말/공휴일 또는 공시 없음)")
        else:
            # ★ 첫 번째 컬럼에 '날짜' 추가 ★
            df.insert(0, "날짜", d.strftime("%Y-%m-%d"))
            all_frames.append(df)
            print(f"{len(df):,}건")
    except Exception as e:
        print(f"오류: {e}")
    time.sleep(1)  # 서버 부하 방지

print(f"\n수집 완료: {len(all_frames)}개 날짜")

## 5. 데이터 통합 및 저장

In [ ]:
if all_frames:
    combined = pd.concat(all_frames, ignore_index=True)
    print(f"통합 결과: {combined.shape[0]:,}행 × {combined.shape[1]}열")
    print("\n컬럼 목록:")
    print(combined.columns.tolist())
    print("\n날짜별 건수:")
    print(combined["날짜"].value_counts().sort_index().to_string())
else:
    print("⚠ 수집된 데이터가 없습니다. DOWNLOAD_URL을 확인하세요.")
    combined = pd.DataFrame()

In [ ]:
if not combined.empty:
    combined.head(10)

In [ ]:
if not combined.empty:
    # Excel 저장
    out_excel = "주요지분현황_2026년3월.xlsx"
    combined.to_excel(out_excel, index=False)
    print(f"Excel 저장 → {out_excel}")

    # CSV 저장 (UTF-8-BOM: 한글 Excel 호환)
    out_csv = "주요지분현황_2026년3월.csv"
    combined.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print(f"CSV   저장 → {out_csv}")
else:
    print("저장할 데이터가 없습니다.")

---

## 참고: 브라우저 DevTools로 정확한 다운로드 URL 확인하기

1. https://fair.krx.co.kr/p/Fids0103/ 접속  
2. **F12** → **Network** 탭 → **XHR** 또는 **All** 필터  
3. 날짜 선택 → **[Excel 다운로드]** 버튼 클릭  
4. Network 탭 목록에서 새로 발생한 요청 클릭  
5. **Headers** 탭 → **Request URL** 전체 복사  
6. **Payload** 탭 → Form 파라미터 확인  
7. 위의 `DOWNLOAD_URL`과 `DATE_PARAM`에 실제 값 입력 후 재실행